# NovaSonic — Three-Agent Trust & Transparency Pipeline (Gemini / Colab) — v2 with auto-retry

Runs a real three-agent pipeline (Researcher → Designer → Communicator) against the assigned NovaSonic challenge, using the **free Google Gemini API**.

This v2 adds **automatic retry/backoff** and short pauses between agents so the free-tier rate limit (429) does not interrupt your recording.

**Steps:** Run Cell 1 (install) → paste your key in Cell 2 and run it → run Cell 3 (the pipeline).

Get a free key (no card): https://aistudio.google.com/app/apikey

In [ ]:
# Cell 1 - Install the Gemini SDK
!pip install -q google-generativeai
print('Gemini SDK installed.')

In [ ]:
# Cell 2 - Paste your free Gemini API key between the quotes, then run this cell.
import google.generativeai as genai

GEMINI_API_KEY = "PASTE_YOUR_KEY_HERE"  # <-- paste your key from https://aistudio.google.com/app/apikey

genai.configure(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"  # confirmed available; if throttled, try 'gemini-2.0-flash-lite'
print('Gemini configured with model:', MODEL)

In [ ]:
# Cell 3 - Run the full three-agent pipeline (with auto-retry on rate limits)
import datetime, time

BUSINESS_CONTEXT = """
BUSINESS: NovaSonic (fictitious podcast platform).
KEY METRICS: 500,000 users; 12,000 creators; creator retention 22% after 6 months;
top 3% of creators generate 81% of all listens.
ASSIGNED LENS: Trust and Transparency.
ASSIGNED CHALLENGE (selected for the pipeline): Mid-tier creators churn because
NovaSonic's monetisation payouts and discovery/recommendation algorithm are an opaque
'black box', and the platform hoards listener data that creators need. The platform
must rebuild creator trust by making economics and discovery transparent and auditable.
"""

RESEARCHER_PROMPT = """You are RESEARCHER-AGENT, the first agent in a three-agent pipeline for NovaSonic.
ROLE (Researcher archetype): Diagnose the trust-and-transparency root causes of creator churn
and specify EXACTLY what transparency data/signals would fix it.
You must:
1. State the single root-cause hypothesis tying the 22% retention and 81% listen concentration
   to opaque platform economics and discovery.
2. Produce a prioritised list of 6-8 concrete, measurable 'transparency data points' that creators
   are currently denied. For each: give the field name, why it builds trust, and the data source.
   Do not invent products or features.
3. Flag 2 risks or unknowns where you are uncertain (be honest about assumptions).
Be specific and concise. Output structured Markdown. Your output will be handed directly to a
DESIGNER-AGENT who will design a dashboard from it, so make the data points unambiguous.
"""

DESIGNER_PROMPT = """You are DESIGNER-AGENT, the second agent in a three-agent pipeline for NovaSonic.
ROLE (Designer archetype): Turn the Researcher's transparency data points into a concrete,
buildable prototype: a 'Creator Trust & Transparency Dashboard'.
CRITICAL: You MUST explicitly reference and use the Researcher's output. Begin your response with a
short section 'HANDOFF RECEIVED' that lists which of the Researcher's data points you are using and
which (if any) you are deferring, with a reason. Do not invent metrics the Researcher did not provide
unless you clearly label them 'NEW (designer-added)' with justification.
Then produce:
1. A text wireframe of the dashboard (sections, panels, key UI components) using Markdown.
2. For each panel, name the exact data point it visualises (traceable to the Researcher).
3. One 'transparency explainer' microcopy example shown to creators (plain language).
4. Acceptance criteria the engineering team could build against.
Output structured Markdown. Your output goes to a COMMUNICATOR-AGENT who will announce this to creators.
"""

COMMUNICATOR_PROMPT = """You are COMMUNICATOR-AGENT, the third and final agent in a three-agent pipeline for NovaSonic.
ROLE (Communicator archetype): Produce the business-actionable final deliverable: a launch
communication to NovaSonic's 12,000 creators announcing the new Creator Trust & Transparency Dashboard.
CRITICAL: You MUST explicitly reference and use the Designer's output. Begin with a short section
'HANDOFF RECEIVED' that names the specific dashboard panels/features (from the Designer) you are
announcing. Do not promise features the Designer did not specify.
Then produce:
1. A creator-facing announcement email (subject line + body) in NovaSonic's honest, non-hyped voice.
2. A short in-app notification (under 280 characters).
3. An FAQ of 4 questions addressing trust concerns (payout fairness, algorithm bias, data access, opt-out).
Be transparent, specific, and avoid marketing exaggeration. Output structured Markdown.
"""

def call_agent(system_prompt, user_content, max_retries=5):
    """Call Gemini with automatic backoff on 429 rate-limit errors."""
    model = genai.GenerativeModel(MODEL, system_instruction=system_prompt)
    delay = 20
    for attempt in range(1, max_retries + 1):
        try:
            resp = model.generate_content(user_content)
            return resp.text
        except Exception as e:
            msg = str(e)
            if ('429' in msg) or ('quota' in msg.lower()) or ('rate' in msg.lower()):
                print(f'   [rate limit hit — waiting {delay}s, then retry {attempt}/{max_retries}...]')
                time.sleep(delay)
                delay = min(delay * 2, 60)
                continue
            raise
    raise RuntimeError('Still rate-limited after retries. Wait a minute and run Cell 3 again, or set MODEL = "gemini-2.0-flash-lite" in Cell 2.')

def banner(text):
    print('\n' + '=' * 75)
    print(text)
    print('=' * 75 + '\n')

PAUSE_BETWEEN_AGENTS = 8  # seconds, spaces out calls to stay under the free per-minute limit

transcript = []
ts = datetime.datetime.now().isoformat()
banner('NOVASONIC THREE-AGENT PIPELINE  |  Researcher -> Designer -> Communicator')
print('Timestamp:', ts)
print('Model:', MODEL)
transcript.append(f'# NovaSonic Three-Agent Pipeline Run\nTimestamp: {ts}\nModel: {MODEL}\n')

# ---- Agent 1: Researcher ----
banner('AGENT 1 / 3  -  RESEARCHER  (diagnosing root cause + transparency data points)')
researcher_out = call_agent(RESEARCHER_PROMPT, BUSINESS_CONTEXT + '\n\nProduce your research brief now.')
print(researcher_out)
transcript.append('## AGENT 1 - RESEARCHER\n### OUTPUT:\n' + researcher_out + '\n')
time.sleep(PAUSE_BETWEEN_AGENTS)

banner('>>> HANDOFF 1: Researcher output -> Designer input <<<')

# ---- Agent 2: Designer ----
banner('AGENT 2 / 3  -  DESIGNER  (turning data points into a dashboard prototype)')
designer_input = (BUSINESS_CONTEXT + '\n\n=== HANDOFF FROM RESEARCHER-AGENT (use this directly) ===\n'
                  + researcher_out + '\n=== END HANDOFF ===\n\nDesign the dashboard now.')
designer_out = call_agent(DESIGNER_PROMPT, designer_input)
print(designer_out)
transcript.append('## AGENT 2 - DESIGNER\n### OUTPUT:\n' + designer_out + '\n')
time.sleep(PAUSE_BETWEEN_AGENTS)

banner('>>> HANDOFF 2: Designer output -> Communicator input <<<')

# ---- Agent 3: Communicator ----
banner('AGENT 3 / 3  -  COMMUNICATOR  (final actionable creator launch package)')
communicator_input = (BUSINESS_CONTEXT + '\n\n=== HANDOFF FROM DESIGNER-AGENT (use this directly) ===\n'
                      + designer_out + '\n=== END HANDOFF ===\n\nWrite the launch communication now.')
communicator_out = call_agent(COMMUNICATOR_PROMPT, communicator_input)
print(communicator_out)
transcript.append('## AGENT 3 - COMMUNICATOR\n### OUTPUT:\n' + communicator_out + '\n')

with open('transcript.md', 'w') as f:
    f.write('\n'.join(transcript))

banner('PIPELINE COMPLETE  -  full transcript saved to transcript.md (see the file panel on the left)')